# CDVAEのMEGNet

In [1]:
import torch
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wandb

%matplotlib inline

In [2]:
result_dir = '/home/fujii/cdvae_comparison/main_results/megnet/'
os.makedirs(result_dir, exist_ok=True)

## CDVAEモデルの学習結果
- lr=0.00100に決定

In [13]:
api = wandb.Api()

In [14]:
urls = [
    'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/ukyvgors',
    'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/171p2zrl',
    'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/czr955zi',
    'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/13w8iowr',

]
lrs = [
    5e-4,
    1e-4,
    1e-3,
    1e-5,
]

In [15]:
all_results = []
for lr, url in zip(lrs, urls):
    run = api.run(url)
    # ログされた履歴を取得
    history = run.history()

    # val_lossの最小値とそのステップを取得
    min_step = history['val_loss'].idxmin()
    min_val_loss = history['val_loss'][min_step]
    val_natom_loss = history['val_natom_loss'][min_step]
    val_natom_accuracy = history['val_natom_accuracy'][min_step]
    val_lattice_loss = history['val_lattice_loss'][min_step]
    val_eform_mae = history['val_eform_mae'][min_step]
    val_gap_mae = history['val_gap_mae'][min_step]
    val_tolerance_acc = history['val_tolerance_acc'][min_step]
    val_100more_acc = history['val_100more_acc'][min_step]

    all_results.append({
        'lr': lr,
        'min_val_loss': min_val_loss,
        'val_natom_loss': val_natom_loss,
        'val_natom_accuracy': val_natom_accuracy,
        'val_lattice_loss': val_lattice_loss,
        'val_eform_mae': val_eform_mae,
        'val_gap_mae': val_gap_mae,
        'val_tolerance_acc': val_tolerance_acc,
        'val_100more_acc': val_100more_acc,
    })

In [16]:
df = pd.DataFrame(all_results)
df.to_csv(os.path.join(result_dir, 'forward_result.csv'), index=False)
display(df)

,lr,min_val_loss,val_natom_loss,val_natom_accuracy,val_lattice_loss,val_eform_mae,val_gap_mae,val_tolerance_acc,val_100more_acc
0,0.00050,16.157167,5.588017,0.357912,0.713992,0.311164,0.883223,0.994828,1.0
1,0.00010,15.265738,5.381971,0.407704,0.623554,0.252162,0.856441,0.992971,1.0
2,0.00100,14.940369,4.739464,0.352438,0.648483,0.282912,0.866748,0.993948,1.0
3,0.00001,20.228668,2.250194,0.443825,0.834301,0.376906,0.938387,0.993755,1.0


## 最適化推論の学習率調整
- 推論コマンドの例: 
    - ```poetry run python scripts/evaluate.py --tasks opt --label lr00001 --lr 0.0001 --num_starting_points 4  --model_path /home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/megnet_lr1e-4/ --target_bg 2.5 --num_saved_crys 0 --megnet_loss_mode True```

In [20]:
result_dir = '/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/megnet_lr1e-4/'
os.listdir(result_dir)

['hparams.yaml',
 'lattice_scaler.pt',
 'wandb',
 '.hydra',
 'eval_opt_lr01000.pt',
 'epoch=2860-step=11444.ckpt',
 'run.log',
 'eval_opt_lr00001.pt',
 'prop_scaler.pt']

In [21]:
d = torch.load(os.path.join(result_dir, 'eval_opt_lr00001.pt'))

/tmp/ipykernel_462473/2581110894.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(os.path.join(result_dir, 'eval_opt_lr00001.pt'))


In [22]:
d['lengths'].shape

torch.Size([1, 4, 3])

In [23]:
d['num_atoms'].shape

torch.Size([1, 4])

In [24]:
d['frac_coords'].shape

torch.Size([1, 33, 3])

In [25]:
d['prediction'] # band gap, formation energy, 100more, tolerance 

tensor([[ 0.0000e+00,  3.7166e-01,  1.2399e-05,  4.4589e-06],
        [ 1.3681e+00, -2.8534e-01,  1.4356e-11,  6.4115e-10],
        [ 1.6054e+00, -1.6310e+00,  8.9640e-11,  5.4235e-10],
        [ 2.4228e+00, -1.3557e+00,  1.6711e-09,  4.5608e-01]])

In [26]:
d['prediction_decoded']

tensor([[ 0.0000e+00,  1.4358e+00,  1.9252e-10,  1.9276e-08],
        [ 0.0000e+00,  8.0926e-01,  4.4942e-12,  1.7332e-10],
        [ 4.6064e-01, -8.8410e-01,  9.6879e-11,  1.1513e-09],
        [ 5.0108e-01, -3.0798e-01,  6.7233e-06,  8.9552e-01]])

In [27]:
d['prediction_matched']

tensor([[ 0.0000e+00,  3.7166e-01,  1.2399e-05,  4.4589e-06],
        [ 1.3681e+00, -2.8534e-01,  1.4356e-11,  6.4115e-10],
        [ 1.6054e+00, -1.6310e+00,  8.9640e-11,  5.4235e-10],
        [ 2.4228e+00, -1.3557e+00,  1.6711e-09,  4.5608e-01]])